# cybersecurity-agent: ISO 27001 RAG Compliance Gap Analysis

This notebook demonstrates the full RAG pipeline:
1. **Download** a public security standards PDF (ISO 27001 summary or NIST CSF fallback)
2. **Ingest & chunk** the PDF text
3. **Build a vector store** with ChromaDB + sentence-transformers
4. **Load synthetic organisational controls**
5. **Run gap analysis** via Claude Haiku for each control
6. **Display results** summary table

**Requirements:**
- `ANTHROPIC_API_KEY` must be set in the environment
- Internet access for PDF download (Cell 2) and Claude API (Cell 6)

⚠️ **HUMAN APPROVAL REQUIRED** before running Cell 2 (PDF download) and Cell 6 (Claude API calls).

In [ ]:
import pathlib
import sys
import os
from IPython.display import display
import pandas as pd

# Locate cybersecurity-agent root regardless of where the notebook is run from
_here = pathlib.Path().resolve()
if (_here / 'cybersecurity-agent').exists():
    CA_ROOT = _here / 'cybersecurity-agent'
elif (_here / 'src').exists() and (_here / 'data').exists():
    CA_ROOT = _here
elif (_here.parent / 'src').exists():
    CA_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate cybersecurity-agent root from {_here}')

SRC = CA_ROOT / 'src'
STANDARDS_DIR = CA_ROOT / 'data' / 'standards'
CONTROLS_DIR = CA_ROOT / 'data' / 'controls'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

STANDARDS_DIR.mkdir(parents=True, exist_ok=True)

print(f'CA_ROOT     : {CA_ROOT}')
print(f'Standards   : {STANDARDS_DIR}')
print(f'Controls    : {CONTROLS_DIR}')
print(f'API key     : {"set" if os.environ.get("ANTHROPIC_API_KEY") else "NOT SET — required for Cell 6"}')
controls = list(CONTROLS_DIR.glob('*.txt'))
print(f'Controls found: {[c.name for c in sorted(controls)]}')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell downloads a PDF from the internet
import requests

ISO_URL = 'https://www.iso.org/files/live/sites/isoorg/files/store/en/PUB100363.pdf'
NIST_URL = 'https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.04162018.pdf'

def download_standard(url: str, dest: pathlib.Path, label: str) -> bool:
    """Attempt to download a PDF; return True on success."""
    try:
        r = requests.get(url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
        if r.status_code == 200 and b'%PDF' in r.content[:10]:
            dest.write_bytes(r.content)
            print(f'Downloaded {label} → {dest} ({len(r.content):,} bytes)')
            return True
        else:
            print(f'{label} returned HTTP {r.status_code} or invalid PDF')
            return False
    except Exception as e:
        print(f'{label} download failed: {e}')
        return False

iso_path = STANDARDS_DIR / 'iso_27001_summary.pdf'
nist_path = STANDARDS_DIR / 'nist_csf.pdf'

if iso_path.exists():
    print(f'ISO PDF already present: {iso_path}')
    STANDARD_PDF = iso_path
    STANDARD_LABEL = 'ISO 27001 Summary'
elif download_standard(ISO_URL, iso_path, 'ISO 27001 Summary'):
    STANDARD_PDF = iso_path
    STANDARD_LABEL = 'ISO 27001 Summary'
elif nist_path.exists():
    print(f'Using existing NIST CSF PDF: {nist_path}')
    STANDARD_PDF = nist_path
    STANDARD_LABEL = 'NIST Cybersecurity Framework'
elif download_standard(NIST_URL, nist_path, 'NIST CSF'):
    STANDARD_PDF = nist_path
    STANDARD_LABEL = 'NIST Cybersecurity Framework'
else:
    raise RuntimeError('Both PDF downloads failed. Check network connectivity.')

print(f'\nUsing standard: {STANDARD_LABEL}')
print(f'PDF path      : {STANDARD_PDF}')

In [ ]:
from ingester import load_pdf, chunk_text

print(f'Loading PDF: {STANDARD_PDF.name} ...')
standard_text = load_pdf(STANDARD_PDF)
print(f'Extracted {len(standard_text):,} characters from {STANDARD_LABEL}')
print(f'\nFirst 500 chars:\n{standard_text[:500]}')

chunks = chunk_text(standard_text, chunk_size=500, overlap=50)
print(f'\nChunked into {len(chunks)} overlapping segments')
print(f'Avg chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars')

In [ ]:
from vectorstore import build_vectorstore

print('Building vector store (downloading all-MiniLM-L6-v2 if not cached) ...')
collection = build_vectorstore(chunks, collection_name='security_standards')
print(f'Vector store built: {collection.count()} chunks indexed')

In [ ]:
control_files = sorted(CONTROLS_DIR.glob('*.txt'))
print(f'Found {len(control_files)} control files:')
for f in control_files:
    text = f.read_text()
    print(f'  {f.name}: {len(text)} chars')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls the Claude API (3 requests, one per control)
from pipeline import run_analysis

print('Running gap analysis via Claude Haiku ...')
results_df = run_analysis(CONTROLS_DIR, collection, k=5)
print(f'\nAnalysis complete: {len(results_df)} controls assessed')
display(results_df[['control_file', 'compliance_level']])

In [ ]:
# Summary: compliance scores and top gaps
level_order = ['compliant', 'partial', 'non-compliant']
level_counts = results_df['compliance_level'].value_counts().reindex(level_order, fill_value=0)

print('=== COMPLIANCE SUMMARY ===')
for level, count in level_counts.items():
    bar = '█' * count
    print(f'  {level:<15} {bar} ({count})')

print('\n=== GAP ANALYSIS BY CONTROL ===')
for _, row in results_df.iterrows():
    print(f'\n[{row["compliance_level"].upper()}] {row["control_file"]}')
    if row['gaps']:
        print('  Gaps:')
        for g in row['gaps']:
            print(f'    - {g}')
    else:
        print('  No gaps identified.')
    if row['recommendations']:
        print('  Recommendations:')
        for r in row['recommendations']:
            print(f'    → {r}')

# All gaps flattened
all_gaps = [g for gaps in results_df['gaps'] for g in gaps]
print(f'\nTotal gaps identified across all controls: {len(all_gaps)}')